# Voice Recognition Model

Objective: This model is aimed to achieve recognition of my voice over any other given voices

## High Level Architecture

![An image of the model architecture](./diagrams/Untitled.png)

## Data Preprocessing

### Dataset Description

This is the infromation about dataset

Folder Structure

- **dataset/**
  - Contains all audio samples.
  - **train/**: Audio files used to train the neural network.
    - **mine/**: Your voice recordings.
    - **others/**: Recordings from other speakers.
  - **validation/**: Audio used to tune hyperparameters and monitor overfitting.
    - **mine/**
    - **others/**
  - **test/**: Unseen audio used to evaluate the final model.
    - **mine/**
    - **others/**

Data Format

- File format: .wav
- Sampling Rate: 16kHZ


In [42]:
# Loading dependencies
import torch
import torchaudio
import torchaudio.transforms as T
from pathlib import Path
import random
import pandas as pd
import subprocess
import os

### Dataset Preprocessing Pipeline

![Data preprocessing pipeline](./diagrams//data-preprocessing.png)

Renaming the raw files to a consistant format

In [3]:
folders = [Path("../data/raw/me/"), Path("../data/raw/others/")]

print("Started Renaming")

for folder in folders:
    files = sorted([f for f in folder.iterdir() if f.is_file()])
    for index, file in enumerate(files, start=1):
        new_name = f"voice_{index:06d}{file.suffix.lower()}"
        new_path = folder / new_name

    file.rename(new_path)

print("Finished renaming.")

Started Renaming
Finished renaming.


Preprocessing the audio

Creating a pipeline

In [77]:
class AudioPreprocessor:
    """ Preprocess audio files to a fixed sample rate and duration."""

    def __init__(self, target_sr, target_seconds):
        self.target_sr = target_sr
        self.target_seconds = target_seconds
        self.target_length = target_sr * target_seconds

    def __call__(self, dataset_directory, convert_to_wav=False):
        return self.process_directory(dataset_directory, convert_to_wav=convert_to_wav)

    def convert_to_wav(self, file, output_dir):
        input_file = Path(file)
        output_file = Path(file).with_suffix(".wav")
        output_dir = Path(output_dir)
        
        command = [
            "ffmpeg",
            "-y", # Overwrite existing files
            "-i", str(file),
            "-ac", "1", # Convert to mono
            "-ar", str(self.target_sr), # Set sample rate
            str(output_dir / output_file.name)
        ]

        if input_file.suffix.lower() == ".wav":
            print(f"File {file} is already a WAV file. Skipping conversion.")
            return output_dir / output_file.name

        try:
            subprocess.run(
                command,
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )

            # Delete original file after successful conversion
            if output_file.exists():
                input_file.unlink()

            print(f"Converted: {file} to {output_dir / output_file.name}")
        except subprocess.CalledProcessError:
            print(f"Failed converting: {file}")
        
        return output_dir / output_file.name

    def preprocess(self, path, convert_to_wav=False):
        # Convert to WAV if needed
        if convert_to_wav:
            new_path = self.convert_to_wav(path, Path(path).parent)
            path = new_path

        waveform, sample_rate = torchaudio.load(path)

        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        print(f"Processing {path}: original sample rate waveform shape {waveform.shape}")

        # Resample
        if sample_rate != self.target_sr:
            waveform = T.Resample(sample_rate, self.target_sr)(waveform)

        print(f"Processing {path}: converted sample rate waveform shape {waveform.shape}")
    
        # Normalize
        waveform = waveform / waveform.abs().max()

        target_length = self.target_sr * self.target_seconds

        # Pad
        if waveform.shape[1] < target_length:
            waveform = torch.nn.functional.pad(
                waveform, (0, target_length - waveform.shape[1]), "constant", 0
            )
        # Trim
        else:
            waveform = waveform[:, :target_length]

        return waveform
    
    def process_directory(self, dataset_directory, convert_to_wav=False):
        me_raw_data = Path(f"{dataset_directory}/raw/me")
        others_raw_data = Path(f"{dataset_directory}/raw/others")

        # me
        for file in me_raw_data.iterdir():
            if file.is_file():
                processed_waveform = self.preprocess(file, convert_to_wav=convert_to_wav)
                # Save or use the processed waveform as needed
                torchaudio.save(
                    f"{dataset_directory}/processed/me/processed_me_{file.name}",
                    processed_waveform,
                    self.target_sr,
                )

        # others
        for file in others_raw_data.iterdir():
            if file.is_file():
                processed_waveform = self.preprocess(file, convert_to_wav=convert_to_wav)
                # Save or use the processed waveform as needed
                torchaudio.save(
                    f"{dataset_directory}/processed/others/processed_others_{file.name}",
                    processed_waveform,
                    self.target_sr,
                )

        self.generate_csv(dataset_directory)

    def check_sample_rates(self, dataset_directory):
        sample_rates = {}
        for root, _, files in os.walk(f"{dataset_directory}/processed"):
            for file in files:
                path = os.path.join(root, file)
                _, sr = torchaudio.load(path)

                sample_rates[sr] = sample_rates.get(sr, 0) + 1

        print(sample_rates)

        if sample_rates.keys() == {self.target_sr}:
            print(f"All files have the target sample rate of {self.target_sr}.")
        else:
            print(f"Some files do not have the target sample rate of {self.target_sr}.")
            for sr, count in sample_rates.items():
                print(f"Sample Rate: {sr}, Count: {count}")
        
    
    def generate_csv(self, dataset_directory):
        DATASET_DIR = Path(f"{dataset_directory}/processed")
        OUTPUT_CSV = Path(f"{dataset_directory}/processed/metadata.csv")

        clip_id = 1
        recording_counter = 1

        TRAIN_RATIO = 0.70
        VALID_RATIO = 0.15
        TEST_RATIO = 0.15

        random.seed(42)

        rows = []

        for class_dir in DATASET_DIR.iterdir():
            
            # Skip if a file
            if class_dir.is_file():
                continue

            files = sorted([f for f in class_dir.iterdir() if f.is_file()])

            random.shuffle(files)

            total = len(files)
            train_end = int(total * TRAIN_RATIO)
            valid_end = train_end + int(total * VALID_RATIO)

            for index, file in enumerate(files):
                if index < train_end:
                    split = "train"
                elif index < valid_end:
                    split = "valid"
                else:
                    split = "test"

                waveform, sr = torchaudio.load(file)

                duration = waveform.shape[1] / sr

                rows.append({
                    "clip_id": clip_id,
                    "recording_id": recording_counter,
                    "filepath": str(file.relative_to(DATASET_DIR)),
                    "speaker_id": class_dir.name,
                    "label": 1 if class_dir.name == "me" else 0,
                    "split": split,
                    "duration": round(duration, 3),
                    "sample_rate": sr
                })

                clip_id += 1
                recording_counter += 1
            
        df = pd.DataFrame(rows)
            
        df.to_csv(OUTPUT_CSV, index=False)

        print(df.head())
        print(f"\nSaved to {OUTPUT_CSV}")


Saving a `data.csv` to save file information instead of having multiple copies for data splits

In [78]:
TARGET_SR = 16000  # Target sample rate
TARGET_SECONDS = 5  # Target duration in seconds

audio_preprocessor = AudioPreprocessor(TARGET_SR, TARGET_SECONDS)

# Process the audio files and generate the CSV metadata
audio_preprocessor("../data/", convert_to_wav=True)

File ../data/raw/me/voice_000007.wav is already a WAV file. Skipping conversion.
Processing ../data/raw/me/voice_000007.wav: original sample rate waveform shape torch.Size([1, 376320])
Processing ../data/raw/me/voice_000007.wav: converted sample rate waveform shape torch.Size([1, 125440])
File ../data/raw/me/voice_000011.wav is already a WAV file. Skipping conversion.
Processing ../data/raw/me/voice_000011.wav: original sample rate waveform shape torch.Size([1, 224256])
Processing ../data/raw/me/voice_000011.wav: converted sample rate waveform shape torch.Size([1, 224256])
File ../data/raw/me/voice_000042.wav is already a WAV file. Skipping conversion.
Processing ../data/raw/me/voice_000042.wav: original sample rate waveform shape torch.Size([1, 928285])
Processing ../data/raw/me/voice_000042.wav: converted sample rate waveform shape torch.Size([1, 928285])
File ../data/raw/me/voice_000047.wav is already a WAV file. Skipping conversion.
Processing ../data/raw/me/voice_000047.wav: origi

Veryfying whether all the audios are in same sample rate

In [65]:
audio_preprocessor.check_sample_rates("../data/")

{16000: 95}
All files have the target sample rate of 16000.


## Feature Extraction

This is how feature extraction pipeline works

![Data preprocessing pipeline](./diagrams/spectogram.png)

Transforming to Mel Spectogram

In [66]:
class MelSpectogramTransform:

    def __init__(self, sample_rate: int = TARGET_SR, n_mels: int = 128, n_fft: int = 1024, hop_length: int = 512):
        
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_mels=n_mels,
            n_fft=n_fft,
            hop_length=hop_length
        )

        self.to_db = T.AmplitudeToDB(stype="power")

    def __call__(self, waveform: torch.Tensor):
        mel = self.mel_transform(waveform)

        # Convert to log-mel

        mel_db = self.to_db(mel)

        return mel_db
    
# Test code
mel_transform = MelSpectogramTransform()
waveform, sr = torchaudio.load("../data/processed/me/processed_me_voice_000003.wav")
test_mel = mel_transform(waveform)
print(test_mel.shape)
print(waveform.shape)

torch.Size([1, 128, 157])
torch.Size([1, 80000])


Scaling the spectogram

In [67]:
class SpectrogramNormalizer:
    def __call__(self, spectrogram):
        self.mean = spectrogram.mean()
        self.std = spectrogram.std()

        return (spectrogram - self.mean) / (self.std + 1e-8)
    
# Test code
spectrogram_normalizer = SpectrogramNormalizer()
mel = spectrogram_normalizer(test_mel)
print(mel.shape)

torch.Size([1, 128, 157])


## Dataset Loader

Reads the `metadata.csv` and convert it to a mel spectrogram and provide the spectrogram and the label

In [ ]:
class DatasetLoader:
    """ Loading dataset from metadata """
    def __init__(self, metadata_path, parent_dir="../data/processed"):
        self.metadata_path = metadata_path

        # Important: It is expected that the metadata CSV file is located in the parent directory of the processed audio files.
        self.metadata = pd.read_csv(f"{parent_dir}/{metadata_path}")
        self.parent_dir = parent_dir

        self.mel_transform = MelSpectogramTransform()
        self.normalizer = SpectrogramNormalizer()

    def __call__(self, audio_path):
        # Label of the current audio file
        label = self.metadata.loc[self.metadata['filepath'] == audio_path, 'label'].iloc[0]

        # Load the audio
        waveform, sr = torchaudio.load(f"{self.parent_dir}/{audio_path}")

        # Mel Spectrogram transformation
        mel_spectrogram = self.mel_transform(waveform)

        # Normalization
        normalized_mel = self.normalizer(mel_spectrogram)

        return normalized_mel, label
    

# Test code
dataset_loader = DatasetLoader("metadata.csv")
normalized_mel, label = dataset_loader("others/processed_others_voice_000001.wav")
print(normalized_mel.shape, label)

torch.Size([1, 128, 157]) 0
